# ML-09 — Validation and Research Claim Audit

This notebook evaluates whether my model's performance holds under a more realistic validation setup.

I compare a random row split with a grouped split by client, inspect possible leakage, review prediction errors, and rewrite claims based on the evidence.

The goal is to report what was observed and measured without making claims that the data cannot support.

## 1. Two Paper Findings and My Methodology Questions

### Finding A

[To be completed from the March 2026 research paper.]

**Question I would ask:** Where does the label for this finding come from, and is it measured after the window the features come from?

**Why it matters:** If the label overlaps the feature window, the result could be partly built in.

**Why I think the authors made this choice:** [To be completed based on the paper.]

### Finding B

[To be completed from the March 2026 research paper.]

**Question I would ask:** Does the validation design (split by client or page) support the claim for pages or clients the model has not seen?

**Why it matters:** A random split can reward memorizing one client's pages.

**What I'd like to see next:** Results by client or by month to check whether the finding holds across different groups and time periods.

I'm asking these questions as practice at the next level of rigor. The goal is to understand how the research could be made stronger, not to criticize the authors.

In [42]:
import duckdb
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download
from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [43]:
TOKEN = userdata.get("HF_TOKEN")

if not TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

REPO = "FlyRank/internship-warehouse"

FACT = hf_hub_download(
    REPO,
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CONTENT = hf_hub_download(
    REPO,
    "dim_content.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CLIENTS = hf_hub_download(
    REPO,
    "dim_clients.parquet",
    repo_type="dataset",
    token=TOKEN
)

con = duckdb.connect()

print("Warehouse files loaded successfully.")

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Warehouse files loaded successfully.


In [44]:
client_history = con.sql(f"""
    SELECT
        client_hash_id,
        gsc_data_start
    FROM read_parquet('{DIM_CLIENTS}')
""").df()

client_history["gsc_data_start"] = pd.to_datetime(
    client_history["gsc_data_start"]
)

window_start = pd.Timestamp("2026-03-01")

print("Total clients in dimension table:", len(client_history))

print(
    "Clients with data starting on or before March 1:",
    (client_history["gsc_data_start"] <= window_start).sum()
)

print(
    "Clients with later or unknown start dates:",
    (client_history["gsc_data_start"] > window_start).sum()
    + client_history["gsc_data_start"].isna().sum()
)

Total clients in dimension table: 104
Clients with data starting on or before March 1: 52
Clients with later or unknown start dates: 52


In [45]:
df = con.sql(f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_early,
        SUM(gsc_clicks) AS clicks_early,
        AVG(gsc_avg_position) AS avg_position_early,

        SUM(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-10'
        ) AS imp_h1,

        SUM(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-10'
        ) AS imp_h2

    FROM read_parquet('{FACT}')

    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'

    GROUP BY 1, 2
),

late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_late

    FROM read_parquet('{FACT}')

    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'

    GROUP BY 1, 2
)

SELECT
    e.client_hash_id,
    e.content_hash_id,

    e.impressions_early,

    e.clicks_early * 1.0 /
        NULLIF(e.impressions_early, 0) AS ctr_early,

    e.avg_position_early,

    (COALESCE(e.imp_h2, 0) + 1.0) /
        (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,

    DATE '2026-03-20' - c.content_updated_date
        AS days_since_update,

    c.search_volume,

    (
        COALESCE(l.clicks_late, 0) / 11.0
        < 0.8 * (e.clicks_early / 20.0)
    ) AS label

FROM early e

JOIN read_parquet('{DIM_CONTENT}') c
    USING (client_hash_id, content_hash_id)

LEFT JOIN late l
    USING (client_hash_id, content_hash_id)

WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'

""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)

print("Dataset shape:", df.shape)

display(
    df.drop(
        columns=["client_hash_id", "content_hash_id"]
    ).head()
)

Dataset shape: (2526, 9)


,impressions_early,ctr_early,avg_position_early,impr_trend_early,days_since_update,search_volume,label
0,2383.0,0.002098,21.429344,1.154472,23.0,0.0,1
1,2343.0,0.005975,8.973752,1.082593,23.0,0.0,1
2,1393.0,0.003589,13.398960,1.200315,23.0,0.0,1
3,7276.0,0.000687,49.787469,0.661265,23.0,0.0,1
4,5482.0,0.000912,30.917178,0.993457,23.0,0.0,1


In [46]:
df["search_volume"] = df["search_volume"].fillna(
    df["search_volume"].median()
)

df["impressions_log"] = np.log1p(
    df["impressions_early"]
)

df["search_volume_log"] = np.log1p(
    df["search_volume"]
)

df["impr_trend_log"] = np.log(
    df["impr_trend_early"].clip(0.05, 20)
)

FEATS = [
    "impressions_log",
    "ctr_early",
    "avg_position_early",
    "days_since_update",
    "search_volume_log",
    "impr_trend_log"
]

LABEL = "label"
GROUP = "client_hash_id"

d = df.dropna(
    subset=[LABEL, GROUP]
).copy().reset_index(drop=True)

print("Final dataset shape:", d.shape)
print("\nMissing feature values:")
print(d[FEATS].isna().sum())

Final dataset shape: (2526, 12)

Missing feature values:
impressions_log       0
ctr_early             0
avg_position_early    0
days_since_update     0
search_volume_log     0
impr_trend_log        0
dtype: int64


In [47]:
def precision_at_k(y, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[order].mean())


def fit_eval(train, test, features=FEATS):

    model = make_pipeline(
        SimpleImputer(strategy="median"),

        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=20,
            random_state=0,
            n_jobs=-1
        )
    )

    model.fit(
        train[features],
        train[LABEL]
    )

    scores = model.predict_proba(
        test[features]
    )[:, 1]

    precision = precision_at_k(
        test[LABEL],
        scores,
        50
    )

    auc = roc_auc_score(
        test[LABEL],
        scores
    )

    return round(precision, 3), round(auc, 3)


print("Evaluation functions are ready.")

Evaluation functions are ready.


In [48]:
# Random row split

tr_a, te_a = train_test_split(
    d,
    test_size=0.3,
    random_state=0
)

before = fit_eval(tr_a, te_a)


# Grouped split by client

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=0
)

tr_i, te_i = next(
    gss.split(d, groups=d[GROUP])
)

tr_g = d.iloc[tr_i].copy()
te_g = d.iloc[te_i].copy()

after = fit_eval(tr_g, te_g)


comparison = pd.DataFrame({
    "Split": [
        "Random rows",
        "Grouped by client"
    ],
    "Precision@50": [
        before[0],
        after[0]
    ],
    "AUC": [
        before[1],
        after[1]
    ],
    "Positive base rate": [
        round(te_a[LABEL].mean(), 3),
        round(te_g[LABEL].mean(), 3)
    ]
})

display(comparison)

print("Random split:", before)
print("Grouped split:", after)

print(
    "Overlapping clients:",
    len(
        set(tr_g[GROUP]) &
        set(te_g[GROUP])
    )
)

,Split,Precision@50,AUC,Positive base rate
0,Random rows,0.68,0.639,0.565
1,Grouped by client,0.64,0.597,0.536


Random split: (0.68, np.float64(0.639))
Grouped split: (0.64, np.float64(0.597))
Overlapping clients: 0


## 2. My Model Under an Honest Split

I compared a random row split with a grouped split by client.

The random split allows records from the same client to appear in both training and testing sets. The grouped split prevents this by keeping each client entirely in one set.

The grouped split tests whether the model can rank records from clients it did not see during training.

Both metrics are reported alongside their positive base rates because Precision@50 should be interpreted relative to the positive rate in the test set.

A lower score under grouped validation may indicate that the random split gave an optimistic estimate. However, the grouped test-set composition must also be considered before making general conclusions.

In [49]:
# Check client separation

train_clients = set(tr_g[GROUP])
test_clients = set(te_g[GROUP])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))


# Check grouped test-set distribution

client_distribution = (
    te_g[GROUP]
    .value_counts()
    .reset_index(drop=True)
)

print("\nGrouped test rows:", len(te_g))
print("Largest client rows:", client_distribution.iloc[0])

print(
    "Largest client share:",
    round(client_distribution.iloc[0] / len(te_g), 3)
)

print("Smallest client rows:", client_distribution.iloc[-1])
print("Median rows per client:", client_distribution.median())

Training clients: 15
Testing clients: 7
Overlapping clients: 0

Grouped test rows: 1400
Largest client rows: 1353
Largest client share: 0.966
Smallest client rows: 2
Median rows per client: 10.0


## 3. Leakage and Feature Timing Audit

The feature window covers March 1–20, 2026, while the target label uses clicks from March 21–31, 2026.

The target is constructed by comparing average daily clicks in the late period with average daily clicks in the early period.

The following features were checked:

| Feature | Time Window | Audit Verdict |
|---|---|---|
| impressions_log | March 1–20 | No direct future leakage identified. |
| ctr_early | March 1–20 | Shares early clicks with the target formula. Potential target-construction dependency. |
| avg_position_early | March 1–20 | No direct future leakage identified. |
| impr_trend_log | March 1–20 | Calculated using two periods within the early window. |
| days_since_update | As of March 20 | Uses the content update date. |
| search_volume_log | Unknown | Exact measurement timestamp is not established. |

No late-period clicks are directly included in the listed feature calculations.

However, `ctr_early` uses `clicks_early`, which is also used to construct the target. This is a shared dependency that deserves further investigation.

The exact timestamp of `search_volume` should also be verified before treating it as a fully time-safe feature.

The dataset also uses a global March window. Client-level history start dates should be checked because different clients may have different amounts of available history.

**Overall verdict:** No direct use of future-period clicks was identified in the features. The shared early-click dependency, unknown search-volume timestamp, and client history differences remain limitations.

In [50]:
rows = []

for feature in FEATS:

    imputer = SimpleImputer(strategy="median")

    x_train = imputer.fit_transform(
        tr_g[[feature]]
    )

    x_test = imputer.transform(
        te_g[[feature]]
    )

    auc = roc_auc_score(
        te_g[LABEL],
        x_test.ravel()
    )

    rows.append({
        "feature": feature,
        "single_feature_AUC": round(auc, 3),
        "AUC_strength": round(max(auc, 1 - auc), 3),
        "direction": (
            "Positive"
            if auc >= 0.5
            else "Inverse"
        )
    })


single_feature_results = pd.DataFrame(rows).sort_values(
    "AUC_strength",
    ascending=False
)

display(single_feature_results)

,feature,single_feature_AUC,AUC_strength,direction
0,impressions_log,0.375,0.625,Inverse
5,impr_trend_log,0.386,0.614,Inverse
2,avg_position_early,0.578,0.578,Positive
4,search_volume_log,0.473,0.527,Inverse
1,ctr_early,0.504,0.504,Positive
3,days_since_update,0.500,0.500,Positive


In [51]:
# Compare the grouped model with and without ctr_early

FEATS_WITHOUT_CTR = [
    feature
    for feature in FEATS
    if feature != "ctr_early"
]

without_ctr = fit_eval(
    tr_g,
    te_g,
    features=FEATS_WITHOUT_CTR
)

ablation_results = pd.DataFrame({
    "Model": [
        "All features",
        "Without ctr_early"
    ],
    "Precision@50": [
        after[0],
        without_ctr[0]
    ],
    "AUC": [
        after[1],
        without_ctr[1]
    ]
})

display(ablation_results)

,Model,Precision@50,AUC
0,All features,0.64,0.597
1,Without ctr_early,0.78,0.606


### Feature Dependency Check

I compared the grouped model using all features with a version that excludes `ctr_early`.

This test helps examine whether the feature adds useful predictive information.

If performance improves after removing it, that suggests the feature may not be useful under this particular split.

However, an improvement after removing a feature does not prove leakage. The result should be interpreted together with the feature's relationship to the target construction.

In [52]:
# Train the grouped model for error analysis

model = make_pipeline(
    SimpleImputer(strategy="median"),

    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

model.fit(
    tr_g[FEATS],
    tr_g[LABEL]
)

te = te_g.copy()

te["score"] = model.predict_proba(
    te[FEATS]
)[:, 1]


# Top 50 predictions

top = te.sort_values(
    "score",
    ascending=False
).head(50)

false_positives = (
    top[top[LABEL] == 0][FEATS + ["score"]]
    .head(5)
)

missed = (
    te[te[LABEL] == 1]
    .sort_values("score", ascending=True)[FEATS + ["score"]]
    .head(5)
)

print(
    "False positives in top 50:",
    int((top[LABEL] == 0).sum())
)

print("\nFalse positive examples:")
display(false_positives.round(3).reset_index(drop=True))

print("\nMissed positive examples:")
display(missed.round(3).reset_index(drop=True))

False positives in top 50: 18

False positive examples:


,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
0,5.943,0.016,5.409,23.0,3.045,-0.745,0.818
1,8.361,0.007,1.905,23.0,0.000,-0.433,0.783
2,8.251,0.003,5.196,23.0,0.000,-1.793,0.773
3,8.577,0.006,6.018,23.0,0.000,-0.465,0.771
4,7.856,0.007,1.513,23.0,0.000,-0.588,0.770



Missed positive examples:


,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
0,7.408,0.007,3.785,23.0,2.398,0.411,0.428
1,7.251,0.013,2.514,23.0,2.398,0.242,0.440
2,7.687,0.005,3.930,23.0,2.398,0.336,0.441
3,7.203,0.006,4.355,23.0,3.045,0.346,0.441
4,7.196,0.018,1.807,23.0,2.398,0.607,0.444


## 4. Failure Analysis

I inspected false positives among the model's top 50 predictions and positive examples that received relatively low prediction scores.

False positives are records ranked highly by the model that did not have the proxy decline label.

Missed positives are records with the proxy decline label that received relatively low scores.

These examples show that the model's ranking is imperfect. A high prediction score does not guarantee the observed outcome, and the model does not identify every positive case.

The examples are descriptive and do not establish general failure patterns across all clients.

### Limitations

- The target is a proxy based on click decline, not a verified judgment that content needs refreshing.
- Random splitting may give an optimistic estimate when records from the same clients appear in both sets.
- The grouped test set may be highly imbalanced across clients.
- The shared use of early clicks in `ctr_early` and the target formula requires further investigation.
- The measurement timestamp for `search_volume` is unclear.
- The global March window may not represent the same history depth for every client.
- The evaluation covers one monthly dataset and does not establish performance across other months or a balanced range of clients.

## 5. Claim Rewrite

| Original Claim | Problem | Safer Version |
|---|---|---|
| "The model predicts which pages need a refresh." | The label is a proxy, not proof of refresh need. | "The model ranks content records associated with the proxy decline label in this sample." |
| "Feature X drives performance." | Association does not establish causation. | "Feature X is associated with the target in this sample." |
| "The model works on any site." | Evaluation covers a limited and potentially imbalanced client sample. | "The model was evaluated on held-out clients, but the grouped test set was dominated by one client." |
| "Removing CTR proves leakage." | Feature ablation does not prove leakage. | "Removing `ctr_early` changed the grouped validation results in this experiment." |

### Final Claim

In this March 2026 dataset, the model showed directional ranking ability for a proxy click-decline label.

Performance under grouped validation was lower than performance under random validation. However, the grouped test set was highly imbalanced across clients.

These results should be treated as exploratory decision-support evidence rather than proof that the model can reliably identify pages requiring a refresh.

## Self-check

- [ ] Two findings from the March 2026 research paper, each with a constructive methodology question.
- [x] Random and grouped validation results reported.
- [x] Positive base rates reported alongside Precision@50.
- [x] Group overlap checked.
- [x] Grouped test-set client imbalance checked.
- [x] Leakage verdict provided for every feature.
- [x] Feature timing and shared early-click dependency discussed.
- [x] Single-feature AUC evaluated on the grouped test set.
- [x] Feature ablation performed.
- [x] False positives and missed positive examples inspected.
- [x] Claims rewritten using cautious, evidence-based language.
- [ ] Notebook runs from top to bottom without errors.
- [ ] No client IDs, content IDs, domains, URLs, private queries, or tokens appear in outputs.
- [ ] Notebook committed under `work/notebooks/w06_validation_audit.ipynb`.